In [1]:
import sys
from helpers import *
if ".." not in sys.path:
    sys.path.append("..")

import cobra
import pandas as pd
from cobra.io import read_sbml_model
import geckopy

model = read_sbml_model("../model/Rpom_05.xml")

# Condition-specific carbon source configuration
# DSS3_early and DSS3_late were sampled from diauxic growth on a mixed
# medium that started at 6:6 mM carbon, corresponding to 3 mM acetate and
# 1 mM glucose. We therefore leave both acetate and glucose exchange open
# for those two conditions.
MEDIA = {
    "DSS3_ac":    {"EX_ac": (-15.01, 1000)},
    "DSS3_glc":   {"EX_glc": (-5.44, 0)},
    "DSS3_late":  {"EX_ac": (-15.01, 1000), "EX_glc": (-5.44, 0)},
    "DSS3_early": {"EX_ac": (-15.01, 1000), "EX_glc": (-5.44, 0)},
}

model

Name,Rpom_05
Memory address,34f6467c0
Number of metabolites,1797
Number of reactions,1799
Number of genes,984
Number of groups,0
Objective expression,1.0*Rpom_hwa_biomass - 1.0*Rpom_hwa_biomass_reverse_5ec2f
Compartments,"c, p, e"


# Using GECKO

This section targets the newer GeckoPy API documented at `geckopy.readthedocs.io`, where enzyme-constrained models are represented with `geckopy.Model` and `geckopy.Protein` objects.

The workflow is:
1. map each omics row onto the SBML gene identifiers already used by the model (`SPO####` and `SPO_RS#####`)
2. combine proteomics with RNA fallback to get one condition-specific score per gene
3. query SABIO-RK by EC number when a reaction annotation supports it, while keeping `DEFAULT_KCAT_PER_S` as the fallback for reactions with no matched kinetic entry
4. convert the COBRA model into a GeckoPy-compatible enzyme-constrained model and split isozyme rules into separate reaction copies
5. allocate a configurable enzyme budget across genes for each condition and solve the resulting ecModels

GeckoPy does not appear to expose a built-in automatic `kcat` retrieval helper, so the notebook uses an external SABIO-RK query workflow and only overrides the placeholder value when a usable hit is found. The omics tables here are relative abundances rather than absolute mmol/gDW measurements, so the concentrations below should be treated as comparative enzyme budgets. Replace `TOTAL_ENZYME_BUDGET_MMOL_GDW` and `DEFAULT_KCAT_PER_S` with calibrated values once curated proteomics and kcat data are available.

In [2]:
from __future__ import annotations

import ast
import csv
import json
import math
import re
from collections import defaultdict
from io import StringIO
from pathlib import Path
from statistics import median
from typing import Dict, FrozenSet, Iterable, List

import cobra
import pandas as pd
import requests
from cobra.core.gene import GPR

try:
    import geckopy
    from geckopy import Protein
    from geckopy.reaction import Reaction as GeckoReaction
except Exception as exc:
    raise RuntimeError(
        "This section expects the newer GeckoPy API (`geckopy.Model`, `geckopy.Protein`). Update your environment before running it."
    ) from exc

OMICS_DIR = Path("../data/clean/omics")
PROT_PATH = OMICS_DIR / "prot.csv"
RNA_PATH = OMICS_DIR / "rna-rel.csv"
KCAT_CACHE_PATH = OMICS_DIR / "sabio_kcat_cache.json"
SABIO_TSV_URL = "https://sabiork.h-its.org/sabioRestWebServices/kineticlawsExportTsv"
TARGET_ORGANISM_TERMS = ("Ruegeria pomeroyi", "Ruegeria pomeroyi DSS-3", "R. pomeroyi")

TOTAL_ENZYME_BUDGET_MMOL_GDW = 0.02
DEFAULT_KCAT_PER_S = 1000.0
DEFAULT_PROTEIN_MW = 50_000.0
MIN_GENE_SCORE = 1e-9
PROTEIN_PREFIX = "prot_"
GECKO_ISOZYME_SUFFIX = "__iso"


def read_omics_rows(path: Path) -> List[Dict[str, str]]:
    with path.open(newline="") as handle:
        return list(csv.DictReader(handle))


def safe_float(value) -> float | None:
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def candidate_gene_ids(row: Dict[str, str]) -> List[str]:
    candidates: List[str] = []
    for column in ("frame_id", "SPO_ID (ACCESSION)"):
        raw_value = row.get(column, "")
        if not raw_value:
            continue
        for token in re.split(r"[,;]", raw_value):
            gene_id = token.strip()
            if gene_id and gene_id not in candidates:
                candidates.append(gene_id)
    return candidates


def condition_abundance_columns(conditions: Iterable[str]) -> Dict[str, str]:
    return {condition: f"{condition}_mean_abund" for condition in conditions}


def build_gene_scores(rows, model_gene_ids, conditions) -> Dict[str, Dict[str, float]]:
    abundances = {condition: defaultdict(list) for condition in conditions}
    columns = condition_abundance_columns(conditions)

    for row in rows:
        matched_gene_ids = [gene_id for gene_id in candidate_gene_ids(row) if gene_id in model_gene_ids]
        if not matched_gene_ids:
            continue

        for condition, column in columns.items():
            value = safe_float(row.get(column))
            if value is None or value <= 0:
                continue
            for gene_id in matched_gene_ids:
                abundances[condition][gene_id].append(value)

    return {
        condition: {
            gene_id: sum(values) / len(values)
            for gene_id, values in gene_map.items()
            if values
        }
        for condition, gene_map in abundances.items()
    }


def merge_protein_and_rna_scores(protein_scores, rna_scores, model_gene_ids):
    merged = {}
    for gene_id in model_gene_ids:
        protein_value = protein_scores.get(gene_id)
        rna_value = rna_scores.get(gene_id)
        if protein_value and rna_value:
            merged[gene_id] = math.sqrt(protein_value * rna_value)
        elif protein_value:
            merged[gene_id] = protein_value
        elif rna_value:
            merged[gene_id] = rna_value
    return merged


def normalize_scores_to_budget(gene_scores, total_budget=TOTAL_ENZYME_BUDGET_MMOL_GDW):
    clipped = {gene_id: value for gene_id, value in gene_scores.items() if value is not None and value > 0}
    total = sum(clipped.values())
    if total <= 0:
        raise ValueError("No positive omics values were available to allocate the enzyme budget.")
    scale = total_budget / total
    return {gene_id: value * scale for gene_id, value in clipped.items()}


def summarize_score_sources(protein_scores, rna_scores, model_gene_ids):
    summary = {"protein+rna": 0, "protein_only": 0, "rna_only": 0, "unconstrained": 0}
    for gene_id in model_gene_ids:
        has_protein = gene_id in protein_scores
        has_rna = gene_id in rna_scores
        if has_protein and has_rna:
            summary["protein+rna"] += 1
        elif has_protein:
            summary["protein_only"] += 1
        elif has_rna:
            summary["rna_only"] += 1
        else:
            summary["unconstrained"] += 1
    return summary


def _dnf_from_gpr_ast(node) -> List[FrozenSet[str]]:
    if isinstance(node, ast.Name):
        return [frozenset([node.id])]

    if isinstance(node, ast.BoolOp):
        child_dnfs = [_dnf_from_gpr_ast(child) for child in node.values]

        if isinstance(node.op, ast.Or):
            variants = []
            for child_dnf in child_dnfs:
                variants.extend(child_dnf)
            return list(dict.fromkeys(variants))

        if isinstance(node.op, ast.And):
            variants = [frozenset()]
            for child_dnf in child_dnfs:
                expanded = []
                for current in variants:
                    for addition in child_dnf:
                        expanded.append(current.union(addition))
                variants = expanded
            return list(dict.fromkeys(variants))

    if hasattr(node, "body"):
        return _dnf_from_gpr_ast(node.body)

    return []


def gpr_to_variants(gpr_rule: str) -> List[FrozenSet[str]]:
    if not gpr_rule or not gpr_rule.strip():
        return []
    tree = GPR.from_string(gpr_rule.strip())
    return _dnf_from_gpr_ast(tree.body)


def protein_id_for_gene(gene_id: str) -> str:
    return gene_id if gene_id.startswith(PROTEIN_PREFIX) else f"{PROTEIN_PREFIX}{gene_id}"


def extract_reaction_ec_numbers(reaction) -> List[str]:
    ec_numbers: List[str] = []
    for source in (getattr(reaction, "annotation", {}) or {}, getattr(reaction, "notes", {}) or {}):
        for key, value in source.items():
            if "ec" not in str(key).lower():
                continue
            values = value if isinstance(value, (list, tuple, set)) else [value]
            for raw_value in values:
                for token in re.split(r"[;,]", str(raw_value)):
                    token = token.strip()
                    if re.fullmatch(r"\d+(?:\.\d+|\.-){3}", token) and token not in ec_numbers:
                        ec_numbers.append(token)
    return ec_numbers


def load_kcat_cache(cache_path: Path = KCAT_CACHE_PATH) -> Dict[str, List[Dict[str, str]]]:
    if cache_path.exists():
        return json.loads(cache_path.read_text())
    return {}


def save_kcat_cache(cache: Dict[str, List[Dict[str, str]]], cache_path: Path = KCAT_CACHE_PATH) -> None:
    cache_path.write_text(json.dumps(cache, indent=2, sort_keys=True))


def query_sabio_kcats_by_ec(ec_number: str, cache: Dict[str, List[Dict[str, str]]]) -> List[Dict[str, str]]:
    if ec_number in cache:
        return cache[ec_number]

    response = requests.post(
        SABIO_TSV_URL,
        params={
            "fields[]": [
                "EntryID",
                "Organism",
                "ECNumber",
                "Parameter",
                "EnzymeType",
                "PubMedID",
                "UniprotID",
            ],
            "q": f'ECNumber:"{ec_number}"',
        },
        timeout=60,
    )
    if response.status_code == 404 or not response.text.strip():
        cache[ec_number] = []
        return cache[ec_number]
    response.raise_for_status()
    table = pd.read_csv(StringIO(response.text), sep="	")
    if table.empty:
        cache[ec_number] = []
        return cache[ec_number]

    table.columns = [str(col).strip() for col in table.columns]
    kcat_rows = table[table["parameter.type"].astype(str).str.lower() == "kcat"].copy()
    if kcat_rows.empty:
        cache[ec_number] = []
        return cache[ec_number]

    records = []
    for row in kcat_rows.to_dict(orient="records"):
        records.append({key: "" if pd.isna(value) else str(value) for key, value in row.items()})
    cache[ec_number] = records
    return records


def coerce_kcat_to_per_second(value: str, unit: str) -> float | None:
    number = safe_float(value)
    if number is None:
        return None
    normalized_unit = unit.replace(" ", "").lower()
    if normalized_unit in {"s^(-1)", "s-1", "1/s", "sec^(-1)"}:
        return number
    if normalized_unit in {"min^(-1)", "min-1", "1/min"}:
        return number / 60.0
    if normalized_unit in {"h^(-1)", "h-1", "1/h"}:
        return number / 3600.0
    return None


def rank_sabio_record(record: Dict[str, str], organism_terms: Iterable[str]) -> tuple:
    organism = record.get("Organism", "")
    organism_lower = organism.lower()
    exact_match = any(term.lower() == organism_lower for term in organism_terms)
    partial_match = any(term.lower() in organism_lower for term in organism_terms)
    wildtype = record.get("EnzymeType", "").strip().lower() == "wildtype"
    has_uniprot = bool(record.get("UniprotID", "").strip())
    has_pubmed = bool(record.get("PubMedID", "").strip())
    return (exact_match, partial_match, wildtype, has_uniprot, has_pubmed)


def choose_kcat_record(records: List[Dict[str, str]], organism_terms: Iterable[str] = TARGET_ORGANISM_TERMS) -> Dict[str, str] | None:
    valid = [
        record
        for record in records
        if coerce_kcat_to_per_second(record.get("parameter.startValue", ""), record.get("parameter.unit", "")) is not None
    ]
    if not valid:
        return None

    best_rank = max(rank_sabio_record(record, organism_terms) for record in valid)
    best = [record for record in valid if rank_sabio_record(record, organism_terms) == best_rank]
    best.sort(key=lambda record: abs(math.log10(coerce_kcat_to_per_second(record["parameter.startValue"], record["parameter.unit"]) or DEFAULT_KCAT_PER_S)))
    return best[0]


def build_reaction_kcat_table(gem, organism_terms: Iterable[str] = TARGET_ORGANISM_TERMS, cache_path: Path = KCAT_CACHE_PATH):
    cache = load_kcat_cache(cache_path)
    reaction_kcats = {}
    provenance = {}

    for reaction in gem.reactions:
        ec_numbers = extract_reaction_ec_numbers(reaction)
        if not ec_numbers:
            continue

        chosen_records = []
        for ec_number in ec_numbers:
            records = query_sabio_kcats_by_ec(ec_number, cache)
            selected = choose_kcat_record(records, organism_terms=organism_terms)
            if selected is not None:
                chosen_records.append((ec_number, selected))

        if not chosen_records:
            continue

        per_second_values = [
            coerce_kcat_to_per_second(record["parameter.startValue"], record["parameter.unit"])
            for _, record in chosen_records
        ]
        per_second_values = [value for value in per_second_values if value is not None]
        if not per_second_values:
            continue

        reaction_kcats[reaction.id] = median(per_second_values)
        provenance[reaction.id] = [
            {
                "ec_number": ec_number,
                "organism": record.get("Organism", ""),
                "enzyme_type": record.get("EnzymeType", ""),
                "pubmed_id": record.get("PubMedID", ""),
                "uniprot_id": record.get("UniprotID", ""),
                "kcat_s": coerce_kcat_to_per_second(record.get("parameter.startValue", ""), record.get("parameter.unit", "")),
            }
            for ec_number, record in chosen_records
        ]

    save_kcat_cache(cache, cache_path)
    return reaction_kcats, provenance


def make_kcat_lookup(reaction_kcats: Dict[str, float], default_kcat_s: float = DEFAULT_KCAT_PER_S):
    def lookup(reaction_id: str, genes_variant: FrozenSet[str]) -> float:
        return reaction_kcats.get(reaction_id, default_kcat_s)

    return lookup


def add_protein_constraints(ec_model, reaction_id: str, genes_variant: FrozenSet[str], kcat_lookup, default_kcat_s=DEFAULT_KCAT_PER_S):
    reaction = ec_model.reactions.get_by_id(reaction_id)
    kcat_s = max(1e-6, float(kcat_lookup(reaction_id, genes_variant)))
    for gene_id in sorted(genes_variant):
        reaction.add_protein(
            protein_id_for_gene(gene_id),
            kcat=kcat_s,
            concentration=None,
            name=gene_id,
        )


def build_gecko_template(gem, kcat_lookup=None, default_kcat_s=DEFAULT_KCAT_PER_S, split_isozymes=True):
    ec_model = geckopy.Model(gem.copy())
    objective_coeffs = cobra.util.solver.linear_reaction_coefficients(gem)
    if kcat_lookup is None:
        kcat_lookup = lambda reaction_id, genes_variant: default_kcat_s

    for reaction in list(gem.reactions):
        variants = gpr_to_variants(reaction.gene_reaction_rule)
        if not variants:
            continue

        original = ec_model.reactions.get_by_id(reaction.id)
        stoichiometry = dict(original.metabolites)
        ec_model.remove_reactions([original])

        reaction_variants = variants if split_isozymes and len(variants) > 1 else [variants[0]]

        for index, genes_variant in enumerate(reaction_variants, start=1):
            copy_id = (
                f"{reaction.id}{GECKO_ISOZYME_SUFFIX}{index}"
                if len(reaction_variants) > 1
                else reaction.id
            )
            reaction_copy = GeckoReaction(copy_id)
            reaction_copy.name = reaction.name
            reaction_copy.subsystem = reaction.subsystem
            reaction_copy.lower_bound = reaction.lower_bound
            reaction_copy.upper_bound = reaction.upper_bound
            reaction_copy.annotation = dict(getattr(reaction, "annotation", {}) or {})
            reaction_copy.notes = dict(getattr(reaction, "notes", {}) or {})
            if len(reaction_variants) > 1:
                reaction_copy.notes["gecko_isozyme_variant"] = sorted(genes_variant)
            reaction_copy.gene_reaction_rule = " and ".join(sorted(genes_variant))
            reaction_copy.add_metabolites(stoichiometry)
            ec_model.add_reactions([reaction_copy])
            if objective_coeffs.get(reaction, 0.0):
                ec_model.reactions.get_by_id(copy_id).objective_coefficient = objective_coeffs[reaction]
            add_protein_constraints(ec_model, copy_id, genes_variant, kcat_lookup=kcat_lookup, default_kcat_s=default_kcat_s)

    for objective_reaction, coefficient in objective_coeffs.items():
        if objective_reaction.id in ec_model.reactions:
            ec_model.reactions.get_by_id(objective_reaction.id).objective_coefficient = coefficient

    return ec_model


def apply_condition_to_ec_model(ec_template, gene_concentrations, medium=None):
    ec_model = ec_template.copy()
    if medium:
        for reaction_id, bounds in medium.items():
            ec_model.reactions.get_by_id(reaction_id).bounds = bounds

    applied = 0
    for gene_id, concentration in gene_concentrations.items():
        protein_id = protein_id_for_gene(gene_id)
        if protein_id in ec_model.proteins:
            ec_model.proteins.get_by_id(protein_id).concentration = concentration
            applied += 1
    return ec_model, applied

In [3]:
GECKO_CONDITIONS = tuple(MEDIA)
MODEL_GENE_IDS = {gene.id for gene in model.genes}

protein_rows = read_omics_rows(PROT_PATH)
rna_rows = read_omics_rows(RNA_PATH)

protein_gene_scores = build_gene_scores(protein_rows, MODEL_GENE_IDS, GECKO_CONDITIONS)
rna_gene_scores = build_gene_scores(rna_rows, MODEL_GENE_IDS, GECKO_CONDITIONS)

combined_gene_scores = {
    condition: merge_protein_and_rna_scores(
        protein_gene_scores[condition],
        rna_gene_scores[condition],
        MODEL_GENE_IDS,
    )
    for condition in GECKO_CONDITIONS
}

gene_concentrations = {
    condition: normalize_scores_to_budget(
        combined_gene_scores[condition],
        total_budget=TOTAL_ENZYME_BUDGET_MMOL_GDW,
    )
    for condition in GECKO_CONDITIONS
}

source_summaries = {
    condition: summarize_score_sources(
        protein_gene_scores[condition],
        rna_gene_scores[condition],
        MODEL_GENE_IDS,
    )
    for condition in GECKO_CONDITIONS
}

reaction_kcats, kcat_provenance = build_reaction_kcat_table(model)
kcat_lookup = make_kcat_lookup(reaction_kcats, default_kcat_s=DEFAULT_KCAT_PER_S)
print(f"SABIO-RK kcat coverage: {len(reaction_kcats)} / {len(model.reactions)} reactions")
print(f"Fallback placeholder kcat ({DEFAULT_KCAT_PER_S:.1f} 1/s) used for {len(model.reactions) - len(reaction_kcats)} reactions")

gecko_template = build_gecko_template(
    model,
    kcat_lookup=kcat_lookup,
    default_kcat_s=DEFAULT_KCAT_PER_S,
    split_isozymes=True,
)

gecko_results = {}
for condition in GECKO_CONDITIONS:
    ec_model_condition, applied_proteins = apply_condition_to_ec_model(
        gecko_template,
        gene_concentrations[condition],
        medium=MEDIA[condition],
    )
    solution_rxn, solution_prot = ec_model_condition.optimize()
    objective_value = solution_rxn.objective_value
    gecko_results[condition] = {
        "model": ec_model_condition,
        "growth": objective_value,
        "reaction_solution": solution_rxn,
        "protein_solution": solution_prot,
        "applied_proteins": applied_proteins,
        "source_summary": source_summaries[condition],
    }

    summary = source_summaries[condition]
    print(
        f"{condition:12s} growth={objective_value:.6f} "
        f"proteins={applied_proteins:4d} "
        f"protein+rna={summary['protein+rna']:4d} "
        f"protein_only={summary['protein_only']:4d} "
        f"rna_only={summary['rna_only']:4d} "
        f"unconstrained={summary['unconstrained']:4d}"
    )

best_condition = max(gecko_results, key=lambda condition: gecko_results[condition]["growth"])
print("Best GECKO condition:", best_condition)
print("Inspect with gecko_results['DSS3_glc']['model'] or gecko_results[best_condition]['model']")

SABIO-RK kcat coverage: 552 / 1799 reactions
Fallback placeholder kcat (1000.0 1/s) used for 1247 reactions
DSS3_ac      growth=0.023810 proteins= 796 protein+rna= 597 protein_only=   0 rna_only= 199 unconstrained= 188
DSS3_glc     growth=0.020860 proteins= 796 protein+rna= 588 protein_only=   0 rna_only= 208 unconstrained= 188
DSS3_late    growth=0.016252 proteins= 796 protein+rna= 604 protein_only=   0 rna_only= 192 unconstrained= 188
DSS3_early   growth=0.029420 proteins= 796 protein+rna= 609 protein_only=   0 rna_only= 187 unconstrained= 188
Best GECKO condition: DSS3_early
Inspect with gecko_results['DSS3_glc']['model'] or gecko_results[best_condition]['model']


In [4]:
condition = "DSS3_glc"
ec_glc = gecko_results[condition]["model"]
solution_rxn = gecko_results[condition]["reaction_solution"]
solution_prot = gecko_results[condition]["protein_solution"]

print(condition, "growth", solution_rxn.objective_value)
print("Top active protein usages:")
print(solution_prot.fluxes.sort_values(ascending=False).head(15))

print("Top active protein-/constrained reactions:")
for reaction_id, flux in solution_rxn.fluxes.abs().sort_values(ascending=False).items():
    reaction = ec_glc.reactions.get_by_id(reaction_id)
    uses_protein = any(met.id.startswith(PROTEIN_PREFIX) for met in reaction.metabolites)
    if uses_protein and abs(solution_rxn.fluxes[reaction_id]) > 1e-6:
        print(f"  {reaction_id:30s} {solution_rxn.fluxes[reaction_id]: .4f}")

# Optional export once the condition-specific ecModel looks reasonable:
# geckopy.io.write_sbml_ec_model(ec_glc, Path("../model/Rpom_05__DSS3_glc__gecko.xml"))

print("Top matched reaction kcats (1/s):")
print(pd.Series(reaction_kcats).sort_values(ascending=False).head(15))


DSS3_glc growth 0.02086001878525776
Top active protein usages:
prot_SPO2157     0.000050
prot_SPOA0315    0.000033
prot_SPO1899     0.000029
prot_SPO2235     0.000020
prot_SPO3071     0.000017
prot_SPO3177     0.000012
prot_SPO1733     0.000012
prot_SPO1366     0.000011
prot_SPO3608     0.000010
prot_SPO2275     0.000009
prot_SPO3891     0.000009
prot_SPOA0444    0.000008
prot_SPOA0355    0.000007
prot_SPO2047     0.000006
prot_SPO2151     0.000004
Name: fluxes, dtype: float64
Top active protein-/constrained reactions:
  ATPSYN-RXN                     -1.4223
  1.10.2.2-RXN                    1.1361
  NADH-DEHYDROG-A-RXN             0.8901
  CYTOCHROME-C-OXIDASE-RXN        0.5694
  PYRUVDEH-RXN__iso1              0.3374
  6PGLUCONOLACT-RXN               0.3170
  PGLUCONDEHYDRAT-RXN             0.3170
  GLU6PDEHYDROG-RXN__iso2         0.3170
  KDPGALDOL-RXN__iso1             0.3170
  MALATE-DEH-RXN                  0.2509
  GLCtpp                          0.2118
  GLUCOKIN-RXN          

In [5]:
GECKO_CONDITIONS = tuple(MEDIA)
MODEL_GENE_IDS = {gene.id for gene in model.genes}

protein_rows = read_omics_rows(PROT_PATH)
rna_rows = read_omics_rows(RNA_PATH)

protein_gene_scores = build_gene_scores(protein_rows, MODEL_GENE_IDS, GECKO_CONDITIONS)
rna_gene_scores = build_gene_scores(rna_rows, MODEL_GENE_IDS, GECKO_CONDITIONS)

combined_gene_scores = {
    condition: merge_protein_and_rna_scores(
        protein_gene_scores[condition],
        rna_gene_scores[condition],
        MODEL_GENE_IDS,
    )
    for condition in GECKO_CONDITIONS
}

gene_concentrations = {
    condition: normalize_scores_to_budget(
        combined_gene_scores[condition],
        total_budget=TOTAL_ENZYME_BUDGET_MMOL_GDW,
    )
    for condition in GECKO_CONDITIONS
}

source_summaries = {
    condition: summarize_score_sources(
        protein_gene_scores[condition],
        rna_gene_scores[condition],
        MODEL_GENE_IDS,
    )
    for condition in GECKO_CONDITIONS
}

gecko_template = build_gecko_template(
    model,
    default_kcat_s=DEFAULT_KCAT_PER_S,
    split_isozymes=True,
)

gecko_results = {}
for condition in GECKO_CONDITIONS:
    ec_model_condition, applied_proteins = apply_condition_to_ec_model(
        gecko_template,
        gene_concentrations[condition],
        medium=MEDIA[condition],
    )
    objective_value = ec_model_condition.slim_optimize()
    gecko_results[condition] = {
        "model": ec_model_condition,
        "growth": objective_value,
        "applied_proteins": applied_proteins,
        "source_summary": source_summaries[condition],
    }

    summary = source_summaries[condition]
    print(
        f"{condition:12s} growth={objective_value:.6f} "
        f"proteins={applied_proteins:4d} "
        f"protein+rna={summary['protein+rna']:4d} "
        f"protein_only={summary['protein_only']:4d} "
        f"rna_only={summary['rna_only']:4d} "
        f"unconstrained={summary['unconstrained']:4d}"
    )

best_condition = max(gecko_results, key=lambda condition: gecko_results[condition]["growth"])
print("Best GECKO condition:", best_condition)
print("Inspect with gecko_results['DSS3_glc']['model'] or gecko_results[best_condition]['model']")

DSS3_ac      growth=0.275548 proteins= 796 protein+rna= 597 protein_only=   0 rna_only= 199 unconstrained= 188
DSS3_glc     growth=0.434411 proteins= 796 protein+rna= 588 protein_only=   0 rna_only= 208 unconstrained= 188
DSS3_late    growth=0.254786 proteins= 796 protein+rna= 604 protein_only=   0 rna_only= 192 unconstrained= 188
DSS3_early   growth=0.320087 proteins= 796 protein+rna= 609 protein_only=   0 rna_only= 187 unconstrained= 188
Best GECKO condition: DSS3_glc
Inspect with gecko_results['DSS3_glc']['model'] or gecko_results[best_condition]['model']


In [6]:
0.434411/0.275548

1.5765347598240596